<h1 style="text-align: center"> Burgers' Equation: </h1>

**Definition:**

Burgers' equation is a 1D model equation that combines **nonlinear convection** (step02) and **diffusion** (step03). It keeps the core structure of the Navier-Stokes momentum equation (nonlinear advection + viscous diffusion) while dropping pressure and the other spatial dimensions.

$$
\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = \nu\frac{\partial^2 u}{\partial x^2}
$$

**Physical meaning:** Newton's 2nd law for a fluid particle.

- **LHS** = material derivative $\frac{Du}{Dt}$, the acceleration of a fluid particle (chain rule with $dx/dt = u$).
  - $\frac{\partial u}{\partial t}$: change measured at a fixed point.
  - $u\frac{\partial u}{\partial x}$: change because the flow carries particles with different velocities past that point.
- **RHS** = net viscous force per unit mass. Only curvature ($\frac{\partial^2 u}{\partial x^2} \neq 0$) produces a net force.

**Key ideas:**

- Each term *adds* its own contribution to $\frac{\partial u}{\partial t}$. The simpler equations are special cases: $\nu = 0$ gives nonlinear convection, and removing the convective term gives diffusion.
- The equation is nonlinear: solutions **cannot** be superposed.
- Inviscid case ($\nu = 0$): $\frac{Du}{Dt} = 0$, so each particle keeps its velocity. Faster fluid catches slower fluid, the front steepens, and a shock forms at $t_b = 1/\max(-u_0')$.
- Viscous case: diffusion is strongest where curvature is large, so the front stays steep but smooth, with thickness $\sim \nu/\Delta u$.
- The conservative form $\frac{\partial}{\partial x}\left(\frac{u^2}{2}\right)$ matters numerically when shocks appear: it gives the correct shock speed.

<h3 style="text-align: center;"> Discretization </h3>

$$
\frac{u_i^{n+1} - u_{i}^{n}}{\Delta t} + u_{i}^{n}\frac{u_{i}^{n} - u_{i-1}^{n}}{\Delta x} = \nu \frac{u_{i+1}^{n}-2u_{i}^{n}+u_{i-1}^{n}}{\Delta x^2}
$$
$$
\downarrow
$$
$$
u_{i}^{n+1} = u_{i}^{n} - u_{i}^{n} \frac{\Delta t}{\Delta x}(u_{i}^{n}-u_{i-1}^{n}) + \nu \frac{\Delta t}{\Delta x^2}(u_{i+1}^{n}-2u_{i}^{n}+u_{i-1}^{n})
$$


<h3 style="text-align: center;"> Initial and Boundary Conditions </h3>

To examine some interesting properties of Burgers' equation, we use different initial and boundary conditions than in the previous steps.

**Initial condition** ($t = 0$):

$$
\begin{aligned}
u &= -\frac{2\nu}{\phi}\frac{\partial \phi}{\partial x} + 4 \\
\phi &= \exp\left(\frac{-x^2}{4\nu}\right) + \exp\left(\frac{-(x-2\pi)^2}{4\nu}\right)
\end{aligned}
$$

**Analytical solution:**

$$
\begin{aligned}
u &= -\frac{2\nu}{\phi}\frac{\partial \phi}{\partial x} + 4 \\
\phi &= \exp\left(\frac{-(x-4t)^2}{4\nu(t+1)}\right) + \exp\left(\frac{-(x-4t-2\pi)^2}{4\nu(t+1)}\right)
\end{aligned}
$$

**Boundary condition** (periodic):

$$
u(0) = u(2\pi)
$$

**Notes:**

- $u = -\frac{2\nu}{\phi}\frac{\partial \phi}{\partial x}$ is the **Cole–Hopf transformation**: it turns Burgers' equation into the heat equation $\frac{\partial \phi}{\partial t} = \nu\frac{\partial^2 \phi}{\partial x^2}$, which can be solved exactly. That is why we have an analytical solution to compare against.
- The resulting profile is a **sawtooth**: $u$ rises linearly (slope 1 at $t = 0$), drops sharply where both Gaussians have the same weight ($x = \pi$ at $t = 0$), and rises again.
- The $+4$ makes the whole pattern travel to the right at speed 4, hence the $x - 4t$. The $(t+1)$ comes from the Gaussians spreading by diffusion: the slope becomes $1/(t+1)$, so the sawtooth slowly flattens.
- **Periodic BC:** the domain wraps around. Whatever leaves through $x = 2\pi$ enters again through $x = 0$.
- With only two Gaussians, the analytical solution is valid only while the front stays inside the domain: $4t + \pi < 2\pi \Rightarrow t < \pi/4$. For longer runs, evaluate it at $\xi = (x - 4t) \bmod 2\pi$ instead of $x - 4t$.


### Code:

#### 1. Analytical solution with SymPy

Instead of differentiating $\phi$ by hand, we let SymPy do it and then turn the result into a normal Python function.

- `sp.symbols('x nu t')` creates symbolic variables: they have a name but no value.
- `phi` is written exactly as in the analytical solution, and `phi.diff(x)` gives $\partial \phi / \partial x$ symbolically. `init_printing(use_latex=True)` makes the notebook render these expressions as LaTeX. `print()` shows the same expression as plain text, which is valid Python: that long line is what we would otherwise have to type by hand.
- `u = -2 * nu * (phiprime/phi) + 4` is the Cole–Hopf transformation, still symbolic.
- `lambdify((t, x, nu), u)` turns the symbolic expression into a numerical function `ufunc(t, x, nu)`. The order of the tuple fixes the order of the arguments. `ufunc(1, 4, 3)` is only a sanity check that it returns a number.
- After `lambdify`, `ufunc` no longer depends on the symbols, so the next cell can reuse the names `x`, `nu` and `t` for the grid, the viscosity value and the time. The symbolic objects are lost from then on: to change the expression, rerun the notebook from the top.

In [ ]:
import numpy as np
import sympy as sp


In [ ]:
from sympy import init_printing
init_printing(use_latex=True)

In [ ]:
x, nu, t = sp.symbols('x nu t')
phi = (sp.exp(-(x - 4*t)**2 / (4*nu*(t + 1)))) + (sp.exp(-(x - 4*t - 2*sp.pi)**2 / (4*nu*(t + 1))))
phi

In [ ]:
phiprime = phi.diff(x)
phiprime

In [ ]:
print(phiprime)

In [ ]:
from sympy.utilities.lambdify import lambdify

u = -2 * nu * (phiprime/phi) + 4
print(u)

In [ ]:
ufunc = lambdify((t, x, nu), u)
print(ufunc(1, 4, 3))

#### 2. Parameters, initial condition and stability

- `x = np.linspace(0, 2*np.pi, nx)` stores **both** ends, $x = 0$ and $x = 2\pi$. With a periodic BC they are the same physical point, so the grid has `nx - 1 = 100` independent points and `dx = 2*np.pi/(nx - 1)`.
- `u` at $t = 0$ is the analytical solution evaluated at `t = 0`, one point at a time. `u_n = np.empty(nx)` only declares the array; the loop overwrites it with `u.copy()`.
- `dt = dx * nu` is the course's choice. The final time is `nt * dt` $\approx 0.44 < \pi/4$, so the two-Gaussian analytical solution is still valid when we compare.

**Is this `dt` stable?** As in step 3, write the scheme as a weighted average of the neighbours, with the local Courant number $C_i = u_i \Delta t/\Delta x$ and $\sigma = \nu \Delta t/\Delta x^2$:

$$
u_i^{n+1} = (1 - C_i - 2\sigma)\,u_i^n + (C_i + \sigma)\,u_{i-1}^n + \sigma\,u_{i+1}^n
$$

All weights are non-negative (no new extrema) if:

- $u_i > 0$, so the backward difference really is upwind. Here $u$ stays between about 1 and 7.
- $C_i + 2\sigma \le 1$: the convection and diffusion limits combined.

With $\Delta t = \nu \Delta x$: $C_{\max} = u_{\max}\,\nu \approx 7 \cdot 0.07 \approx 0.49$ and $\sigma = \nu^2/\Delta x \approx 0.078$, so $C_{\max} + 2\sigma \approx 0.65 \le 1$. Stable.

**Careful:** $\sigma = \nu^2/\Delta x$ **grows** when the mesh is refined. With $\Delta t = \nu\Delta x$ the limit $C_{\max} + 2\sigma = 1$ is reached around `nx` $\approx 330$, and finer meshes become unstable. To refine the mesh, compute `dt` from the stability limits instead, e.g. `dt = min(C * dx / u_max, sigma * dx**2 / nu)`.

In [ ]:
from matplotlib import pyplot as plt
%matplotlib inline

nx = 101
nt = 100
dx = 2 * np.pi / (nx - 1)
nu = .07
dt = dx * nu

x = np.linspace(0, 2 * np.pi, nx)
u_n = np.empty(nx)
t = 0

u = np.asarray([ufunc(t, x0, nu) for x0 in x])
u

In [ ]:
plt.figure(figsize=(8, 6), dpi=100)
plt.plot(x, u, marker='o', lw=2)
plt.xlim([0, 2*np.pi])
plt.ylim([0, 10])

#### 3. Time loop and periodic boundary condition

- Interior points `i = 1 ... nx-2` use the discretized equation directly: backward (upwind) difference for the convective term, central difference for the diffusive term, as in steps 2 and 3.
- Point `i = 0` has no left neighbour inside the array. Periodic means that its left neighbour is the point just before $x = 2\pi$, which is `u_n[-2]`. It is **not** `u_n[-1]`: that is $x = 2\pi$, the same physical point as $x = 0$, so the point would use itself as its left neighbour (the convective difference would vanish and the diffusive term would be wrong).
- `u[-1] = u[0]` keeps the duplicated end point equal to the first one. So for the last interior point, `i = nx-2`, the right neighbour `u_n[nx-1]` is the value at $x = 0$, which is consistent with the wrap-around.
- `u_sol` is the analytical solution at the same final time, `t = nt * dt`.

In [ ]:
for n in range (nt):
    u_n = u.copy()
    for i in range(1, nx-1):
        u[i] = u_n[i] - u_n[i] * dt/dx * (u_n[i] - u_n[i-1]) + nu * dt/(dx**2) * (u_n[i + 1] - 2*u_n[i] + u_n[i-1])
    u[0] = u_n[0] - u_n[0] * (dt/dx) * (u_n[0] - u_n[-2]) + nu * dt/(dx**2) * (u_n[1] - 2*u_n[0] + u_n[-2])
    u[-1] = u[0]
u_sol = np.asarray([ufunc(nt*dt, xi, nu) for xi in x])


In [ ]:
plt.figure(figsize=(11, 7), dpi=100)
plt.plot(x,u, marker='o', lw=2, label='Computational')
plt.plot(x, u_sol, label='Analytical')
plt.xlim([0, 2 * np.pi])
plt.ylim([0, 10])
plt.legend();

## Response

### What the simulation shows

- **The sawtooth travels to the right.** The $+4$ moves the whole pattern at speed 4: the front starts at $x = \pi$ and is at $x = \pi + 4t \approx 4.9$ at the final time $t \approx 0.44$. What leaves through $x = 2\pi$ comes back in through $x = 0$, which is why the left part of the plot is still a ramp.
- **The ramps flatten and the jump shrinks.** Along a ramp the fluid ahead is faster than the fluid behind, so the ramp stretches (like the rarefaction in step 2) and its slope decreases as $1/(t+1)$. The jump goes from $\Delta u = 2\pi$ at $t = 0$ to $\Delta u = 2\pi/(t+1) \approx 4.4$, and the kinetic energy lost is dissipated by viscosity inside the front.
- **The front stays steep.** Faster fluid behind the front keeps catching up with slower fluid ahead (nonlinear steepening), while diffusion smooths it. The balance gives a $\tanh$ profile across the front, $\tanh\left(\Delta u\,(x - x_s)/4\nu\right)$, of thickness $\delta = 4\nu/\Delta u \approx 0.06$. That is about **one grid spacing** ($\Delta x \approx 0.063$), so on this mesh the front behaves almost like a shock.
- Away from the front the numerical and analytical solutions agree well: the ramps are smooth and the scheme handles them easily.

### Why the numerical front does not match the analytical one

At the front the numerical solution is wrong in three ways: it lags behind (by about 0.25, around 4 cells), it is about twice as wide, and its peak is lower. There are two separate causes.

**1. The non-conservative form gives the wrong front speed.** A jump must move at the Rankine–Hugoniot speed, which comes from the flux (conservative) form of the equation:

$$
\frac{\partial u}{\partial t} + \frac{\partial}{\partial x}\left(\frac{u^2}{2}\right) = \nu\frac{\partial^2 u}{\partial x^2}
\quad\Rightarrow\quad
s = \frac{u_L + u_R}{2} = 4
$$

The scheme discretizes $u\,\partial u/\partial x$ as $u_i(u_i - u_{i-1})$. Summed over the grid these terms do not cancel in pairs (they do not telescope), so $\sum_i u_i$ is not conserved across a front that spans only a few cells, and the front moves at a speed set by the discretization instead of $s$. Here it moves too slowly. This is the same effect as in step 2. The difference is that here $\nu > 0$, so the exact solution is smooth: refining the mesh until $\Delta x \ll \delta$ does remove the error, but only at first order (halving $\Delta x$ roughly halves the lag).

**2. Upwind adds numerical diffusion.** The backward difference behaves like an extra viscosity

$$
\nu_{num} \approx \frac{u\,\Delta x}{2}\,(1 - C) \approx 0.09 \quad \text{at } u \approx 4,\; C \approx 0.28
$$

which is larger than the physical $\nu = 0.07$. The scheme effectively solves Burgers' equation with more than twice the viscosity, so the front is about twice as thick and the peak is lower.

**How to check it:**

- The mean value `u[:-1].mean()` should stay at 4 (periodic domain, nothing enters or leaves). With this scheme it decreases.
- Replacing the convective term by the conservative flux difference, `dt/dx * (u_n[i]**2/2 - u_n[i-1]**2/2)`, should put the front in the right place, while the peak stays slightly low because the numerical diffusion is still there. Only refining the mesh (with a stable `dt`) reduces that part.